# Polymer Five-Seed Core Study

This notebook runs the polymer core RL methods across the configured study seeds, saves the native per-run bundles/plots, loads the existing baseline MPC save from `Data/`, and exports aggregate slide-ready analysis artifacts.


## Study Scope

- Methods: `horizon_dueling`, `matrix`, `weights`, `residual`, `combined`
- Continuous methods: TD3 only
- Discrete methods: dueling DQN only
- Combined override: dueling DQN horizon + TD3 matrix/weights/residual
- Baseline MPC: imported from the canonical polymer `Data/` save; never rerun here


In [1]:
from pprint import pprint

from utils.polymer_multiseed_core_study import default_study_config, run_polymer_multiseed_core_study

STUDY = default_study_config()

# Notebook-facing overrides.
METHODS = list(STUDY["methods"])
SEEDS = list(STUDY["seeds"])
PARALLEL_METHODS = bool(STUDY.get("parallel_methods", True))
PARALLEL_SEEDS = bool(STUDY.get("parallel_seeds", True))
PARALLEL_METHOD_WORKERS = STUDY.get("max_method_workers")
PARALLEL_RUN_WORKERS = STUDY.get("max_run_workers")
SAVE_PDF = bool(STUDY["save_pdf"])
STYLE_PROFILE = STUDY["style_profile"]
FIGURE_PREFIX = STUDY["figure_prefix"]
REWARD_TAIL_EPISODES = int(STUDY["reward_tail_episodes"])
POLYMER_DATA_DIR_OVERRIDE = STUDY["data_dir_override"]
POLYMER_RESULTS_DIR_OVERRIDE = STUDY["results_dir_override"]


In [2]:
study_config_snapshot = {
    "methods": METHODS,
    "seeds": SEEDS,
    "parallel_methods": PARALLEL_METHODS,
    "parallel_seeds": PARALLEL_SEEDS,
    "max_method_workers": PARALLEL_METHOD_WORKERS,
    "max_run_workers": PARALLEL_RUN_WORKERS,
    "save_pdf": SAVE_PDF,
    "style_profile": STYLE_PROFILE,
    "figure_prefix": FIGURE_PREFIX,
    "reward_tail_episodes": REWARD_TAIL_EPISODES,
    "data_dir_override": POLYMER_DATA_DIR_OVERRIDE,
    "results_dir_override": POLYMER_RESULTS_DIR_OVERRIDE,
}

pprint(study_config_snapshot)


{'data_dir_override': None,
 'figure_prefix': 'polymer_five_seed_core_study',
 'max_method_workers': None,
 'max_run_workers': None,
 'methods': ['horizon_dueling', 'matrix', 'weights', 'residual', 'combined'],
 'parallel_methods': True,
 'parallel_seeds': True,
 'results_dir_override': None,
 'reward_tail_episodes': 20,
 'save_pdf': False,
 'seeds': [7, 11, 23],
 'style_profile': 'paper'}


In [3]:
from pathlib import Path

CONTINUE_ON_ERROR = True
METHOD_RESULTS = {}

def run_method_cell(method_key):
    result = run_polymer_multiseed_core_study(
        methods=[method_key],
        seeds=SEEDS,
        data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
        results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
        save_pdf=SAVE_PDF,
        style_profile=STYLE_PROFILE,
        figure_prefix=f"{FIGURE_PREFIX}_{method_key}",
        reward_tail_episodes=REWARD_TAIL_EPISODES,
        continue_on_error=CONTINUE_ON_ERROR,
        parallel_methods=False,
        parallel_seeds=PARALLEL_SEEDS,
        max_run_workers=(PARALLEL_RUN_WORKERS or len(SEEDS)) if PARALLEL_SEEDS else None,
    )
    METHOD_RESULTS[method_key] = result
    failed_runs = result.get("failed_runs", [])
    print({
        "method": method_key,
        "parallel_seeds": bool(result.get("parallel_seeds")),
        "parallel_worker_count": result.get("parallel_worker_count"),
        "submitted_task_count": result.get("submitted_task_count"),
        "successful_runs": len(result.get("run_records", [])),
        "failed_runs": len(failed_runs),
        "summary_markdown": result.get("summary_markdown"),
    })
    if failed_runs:
        print("Failed runs saved to:", result.get("failed_runs_csv"))
        for failure in failed_runs:
            print(f"seed={failure['seed']} :: {failure['error_type']} :: {failure['error_message']}")
    return result

def run_all_methods_cell(*, parallel_methods):
    result = run_polymer_multiseed_core_study(
        methods=METHODS,
        seeds=SEEDS,
        data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
        results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
        save_pdf=SAVE_PDF,
        style_profile=STYLE_PROFILE,
        figure_prefix=FIGURE_PREFIX,
        reward_tail_episodes=REWARD_TAIL_EPISODES,
        continue_on_error=CONTINUE_ON_ERROR,
        parallel_methods=parallel_methods,
        parallel_seeds=PARALLEL_SEEDS,
        max_method_workers=(PARALLEL_METHOD_WORKERS or len(METHODS)) if parallel_methods else None,
        max_run_workers=PARALLEL_RUN_WORKERS,
    )
    print({
        "parallel_methods": bool(result.get("parallel_methods")),
        "parallel_seeds": bool(result.get("parallel_seeds")),
        "max_method_workers": result.get("max_method_workers"),
        "max_run_workers": result.get("max_run_workers"),
        "parallel_worker_count": result.get("parallel_worker_count"),
        "submitted_task_count": result.get("submitted_task_count"),
        "successful_runs": len(result.get("run_records", [])),
        "failed_runs": len(result.get("failed_runs", [])),
        "summary_markdown": result.get("summary_markdown"),
    })
    return result

def show_method_summary(method_key):
    result = METHOD_RESULTS[method_key]
    summary_path = Path(result["summary_markdown"])
    print(summary_path)
    print()
    print(summary_path.read_text(encoding="utf-8"))


In [4]:
horizon_dueling_result = run_method_cell("horizon_dueling")
horizon_dueling_result


c:\Users\HAMEDI\Desktop\RL_assisted_MPC\utils\polymer_multiseed_core_study.py:172: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=0))
c:\Users\HAMEDI\miniconda3\envs\rl\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


{'method': 'horizon_dueling', 'parallel_seeds': True, 'parallel_worker_count': 3, 'submitted_task_count': 3, 'successful_runs': 3, 'failed_runs': 0, 'summary_markdown': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_horizon_dueling\\20260503_151836\\polymer_five_seed_core_summary.md'}


{'output_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_horizon_dueling\\20260503_151836',
 'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Data\\mpc_results_dist.pickle',
 'methods': ['horizon_dueling'],
 'seeds': [7, 11, 23],
 'parallel_methods': False,
 'parallel_seeds': True,
 'max_method_workers': None,
 'max_run_workers': 3,
 'parallel_worker_count': 3,
 'submitted_task_count': 3,
 'run_records': [{'method_key': 'horizon_dueling',
   'method_label': 'Dueling DQN Horizon',
   'seed': 7,
   'bundle_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\dueling_horizon_disturb_unified_seed007\\20260503_165856\\input_data.pkl',
   'result_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\dueling_horizon_disturb_unified_seed007\\20260503_165856',
   'compare_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\disturb_compare_dueling_horizon_unified_seed007\\

In [5]:
matrix_result = run_method_cell("matrix")
matrix_result


{'method': 'matrix', 'parallel_seeds': True, 'parallel_worker_count': 3, 'submitted_task_count': 3, 'successful_runs': 3, 'failed_runs': 0, 'summary_markdown': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_matrix\\20260503_170045\\polymer_five_seed_core_summary.md'}


{'output_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_matrix\\20260503_170045',
 'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Data\\mpc_results_dist.pickle',
 'methods': ['matrix'],
 'seeds': [7, 11, 23],
 'parallel_methods': False,
 'parallel_seeds': True,
 'max_method_workers': None,
 'max_run_workers': 3,
 'parallel_worker_count': 3,
 'submitted_task_count': 3,
 'run_records': [{'method_key': 'matrix',
   'method_label': 'TD3 Matrix',
   'seed': 7,
   'bundle_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\td3_multipliers_disturb_seed007\\20260503_173906\\input_data.pkl',
   'result_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\td3_multipliers_disturb_seed007\\20260503_173906',
   'compare_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\disturb_compare_td3_multipliers_seed007\\20260503_173922',
   'baseline_path': 'C:\\Users\\HAMEDI\\De

In [6]:
weights_result = run_method_cell("weights")
weights_result


{'method': 'weights', 'parallel_seeds': True, 'parallel_worker_count': 3, 'submitted_task_count': 3, 'successful_runs': 3, 'failed_runs': 0, 'summary_markdown': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_weights\\20260503_173957\\polymer_five_seed_core_summary.md'}


{'output_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_weights\\20260503_173957',
 'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Data\\mpc_results_dist.pickle',
 'methods': ['weights'],
 'seeds': [7, 11, 23],
 'parallel_methods': False,
 'parallel_seeds': True,
 'max_method_workers': None,
 'max_run_workers': 3,
 'parallel_worker_count': 3,
 'submitted_task_count': 3,
 'run_records': [{'method_key': 'weights',
   'method_label': 'TD3 Weights',
   'seed': 7,
   'bundle_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\td3_weights_disturb_seed007\\20260503_181915\\input_data.pkl',
   'result_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\td3_weights_disturb_seed007\\20260503_181915',
   'compare_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\disturb_compare_td3_weights_seed007\\20260503_181928',
   'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\R

In [7]:
residual_result = run_method_cell("residual")
residual_result


{'method': 'residual', 'parallel_seeds': True, 'parallel_worker_count': 3, 'submitted_task_count': 3, 'successful_runs': 3, 'failed_runs': 0, 'summary_markdown': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_residual\\20260503_181944\\polymer_five_seed_core_summary.md'}


{'output_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_residual\\20260503_181944',
 'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Data\\mpc_results_dist.pickle',
 'methods': ['residual'],
 'seeds': [7, 11, 23],
 'parallel_methods': False,
 'parallel_seeds': True,
 'max_method_workers': None,
 'max_run_workers': 3,
 'parallel_worker_count': 3,
 'submitted_task_count': 3,
 'run_records': [{'method_key': 'residual',
   'method_label': 'TD3 Residual',
   'seed': 7,
   'bundle_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\td3_residual_disturb_seed007\\20260503_185904\\input_data.pkl',
   'result_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\td3_residual_disturb_seed007\\20260503_185904',
   'compare_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\disturb_compare_td3_residual_seed007\\20260503_185922',
   'baseline_path': 'C:\\Users\\HAMEDI\\Des

In [8]:
combined_result = run_method_cell("combined")
combined_result


c:\Users\HAMEDI\Desktop\RL_assisted_MPC\utils\polymer_multiseed_core_study.py:172: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=0))
c:\Users\HAMEDI\miniconda3\envs\rl\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


{'method': 'combined', 'parallel_seeds': True, 'parallel_worker_count': 3, 'submitted_task_count': 3, 'successful_runs': 3, 'failed_runs': 0, 'summary_markdown': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_combined\\20260503_185939\\polymer_five_seed_core_summary.md'}


{'output_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study_combined\\20260503_185939',
 'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Data\\mpc_results_dist.pickle',
 'methods': ['combined'],
 'seeds': [7, 11, 23],
 'parallel_methods': False,
 'parallel_seeds': True,
 'max_method_workers': None,
 'max_run_workers': 3,
 'parallel_worker_count': 3,
 'submitted_task_count': 3,
 'run_records': [{'method_key': 'combined',
   'method_label': 'Combined',
   'seed': 7,
   'bundle_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\combined_disturb_h_dueling_dqn_mismatch__m_td3_mismatch__w_td3_mismatch__r_td3_mismatch_rho_seed007\\20260503_214519\\input_data.pkl',
   'result_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\combined_disturb_h_dueling_dqn_mismatch__m_td3_mismatch__w_td3_mismatch__r_td3_mismatch_rho_seed007\\20260503_214519',
   'compare_dir': 'C:\\Users\\HAMEDI\\Deskt

In [9]:
for method_key in METHODS:
    if method_key not in METHOD_RESULTS:
        continue
    print(f"===== {method_key} =====")
    show_method_summary(method_key)
    print()


===== horizon_dueling =====
C:\Users\HAMEDI\Desktop\RL_assisted_MPC\report\figures\polymer_five_seed_core_study_horizon_dueling\20260503_151836\polymer_five_seed_core_summary.md

# Polymer Five-Seed Core Study

## Method Summary

| Method | Final reward | Tail reward | RMSE mean | IAE mean | Tail offset | Mean |Δu| | ΔRMSE vs MPC |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Dueling DQN Horizon | -2.599 +/- 0.054 | -2.623 +/- 0.006 | 0.293 +/- 0.001 | 7499.786 +/- 75.058 | 0.086 +/- 0.008 | 1.259 +/- 0.005 | -0.087 +/- 0.001 |

## Baseline Reference

- Baseline path: `C:\Users\HAMEDI\Desktop\RL_assisted_MPC\Polymer\Data\mpc_results_dist.pickle`
- RMSE mean: 0.3795
- IAE mean: 13193.5618
- Tail offset mean: 0.1650
- Mean |Δu|: 0.7140


===== matrix =====
C:\Users\HAMEDI\Desktop\RL_assisted_MPC\report\figures\polymer_five_seed_core_study_matrix\20260503_170045\polymer_five_seed_core_summary.md

# Polymer Five-Seed Core Study

## Method Summary

| Method | Final reward | Tail rewa

In [10]:
study_result = run_all_methods_cell(parallel_methods=PARALLEL_METHODS)

study_result


c:\Users\HAMEDI\Desktop\RL_assisted_MPC\utils\polymer_multiseed_core_study.py:172: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=0))
c:\Users\HAMEDI\miniconda3\envs\rl\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


{'parallel_methods': True, 'parallel_seeds': True, 'max_method_workers': 5, 'max_run_workers': None, 'parallel_worker_count': 15, 'submitted_task_count': 15, 'successful_runs': 15, 'failed_runs': 0, 'summary_markdown': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study\\20260503_214702\\polymer_five_seed_core_summary.md'}


{'output_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\report\\figures\\polymer_five_seed_core_study\\20260503_214702',
 'baseline_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Data\\mpc_results_dist.pickle',
 'methods': ['horizon_dueling', 'matrix', 'weights', 'residual', 'combined'],
 'seeds': [7, 11, 23],
 'parallel_methods': True,
 'parallel_seeds': True,
 'max_method_workers': 5,
 'max_run_workers': None,
 'parallel_worker_count': 15,
 'submitted_task_count': 15,
 'run_records': [{'method_key': 'horizon_dueling',
   'method_label': 'Dueling DQN Horizon',
   'seed': 7,
   'bundle_path': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\dueling_horizon_disturb_unified_seed007\\20260504_004212\\input_data.pkl',
   'result_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\dueling_horizon_disturb_unified_seed007\\20260504_004212',
   'compare_dir': 'C:\\Users\\HAMEDI\\Desktop\\RL_assisted_MPC\\Polymer\\Results\\disturb_compare_due

In [11]:
study_result_sequential = run_all_methods_cell(parallel_methods=False)

study_result_sequential


: 

: 